# 🎯 Module 8 — NVIDIA Interview Prep: 100 Questions

> **Series:** GPU Programming with Python & CUDA  
> **Coverage:** All NVIDIA roles from Modules 1–7

This is an **interactive quiz**. For each question:
1. Read the question and formulate your answer mentally (or type it)
2. Click **"Evaluate with AI"** to get Claude's assessment of your answer
3. Or click **"Show Answer"** to reveal the model answer directly

### 📋 Categories
| # | Category | Role |
|---|---|---|
| 1–15 | CUDA Fundamentals | All GPU roles |
| 16–30 | Memory Management | All GPU roles |
| 31–45 | Warp-Level Programming | All GPU roles |
| 46–57 | Graphics Compiler & Testing | SDET — Graphics Compiler |
| 58–68 | DL Compiler Verification | Senior DL Compiler Engineer |
| 69–78 | Math Libraries & LLM | Senior Math Libraries Engineer |
| 79–100 | Inference & Model Optimization | Senior DL Software Engineer |

In [1]:
import os
from IPython.display import display, HTML, clear_output

try:
    import ipywidgets as widgets
    WIDGETS_AVAILABLE = True
except Exception:
    widgets = None
    WIDGETS_AVAILABLE = False

# Anthropic is optional: the quiz should still run without AI evaluation.
try:
    import anthropic
except Exception:
    anthropic = None

# Anthropic client — picks up ANTHROPIC_API_KEY from environment automatically
try:
    _client = anthropic.Anthropic() if anthropic else None
    AI_AVAILABLE = _client is not None
except Exception:
    _client = None
    AI_AVAILABLE = False

print("✅ Widgets ready" if WIDGETS_AVAILABLE else "⚠️ ipywidgets not installed; using text-mode quiz fallback")
print(f"{'✅' if AI_AVAILABLE else '⚠️ '} AI evaluation {'available' if AI_AVAILABLE else 'unavailable (install anthropic and/or set ANTHROPIC_API_KEY) — use Show Answer instead'}")

✅ Widgets ready
✅ AI evaluation available


In [2]:
QUESTIONS = [
    # ── CUDA Fundamentals (1–15) ──────────────────────────────────────────────
    {
        "n": 1, "category": "CUDA Fundamentals",
        "q": "What is the relationship between a CUDA thread, block, and grid?",
        "a": "A thread is the smallest unit of execution. Threads are grouped into blocks, which share shared memory and can synchronise with __syncthreads(). Blocks are grouped into a grid, which is the entire kernel launch. Threads cannot synchronise across blocks (without cooperative groups). The hierarchy maps to hardware: threads → warp → SM (block) → GPU (grid)."
    },
    {
        "n": 2, "category": "CUDA Fundamentals",
        "q": "What is a warp, and why does NVIDIA use exactly 32 threads?",
        "a": "A warp is a group of 32 threads that execute the same instruction in lockstep (SIMT). 32 was chosen as a balance between hardware area for the warp scheduler, register file width, and memory transaction granularity (a 128-byte cache line holds 32 float32 values). It has remained 32 across all GPU generations for binary compatibility."
    },
    {
        "n": 3, "category": "CUDA Fundamentals",
        "q": "How do you compute a thread's global index in a 1D kernel?",
        "a": "global_idx = blockIdx.x * blockDim.x + threadIdx.x. You also guard out-of-bounds accesses: if (global_idx >= n) return; This pattern works because blockIdx.x tells you which block you're in, blockDim.x is the stride between blocks, and threadIdx.x is your position within the block."
    },
    {
        "n": 4, "category": "CUDA Fundamentals",
        "q": "What are the CUDA function qualifiers __global__, __device__, and __host__?",
        "a": "__global__: called from CPU, runs on GPU — this is a kernel. __device__: called from GPU, runs on GPU — a helper function. __host__: called from CPU, runs on CPU (default). You can combine __host__ __device__ to compile the same function for both. __global__ functions must return void and cannot be recursive (without special support)."
    },
    {
        "n": 5, "category": "CUDA Fundamentals",
        "q": "What is SIMT execution and how does it differ from SIMD?",
        "a": "SIMT (Single Instruction, Multiple Threads) executes one instruction across 32 threads (a warp), each with its own registers and program counter. Unlike CPU SIMD where one instruction operates on a fixed-width vector register, SIMT threads can independently branch — the hardware masks inactive threads and serialises divergent paths. SIMT gives the illusion of independent threads while executing in lockstep."
    },
    {
        "n": 6, "category": "CUDA Fundamentals",
        "q": "What does __syncthreads() do, and what happens if you use it incorrectly?",
        "a": "__syncthreads() is a block-level barrier — all threads in the block must reach it before any can proceed. It is typically used after writing to shared memory to ensure all writes are visible before reads. If threads in a warp diverge and only some reach __syncthreads(), it causes a deadlock because the barrier waits for all threads unconditionally."
    },
    {
        "n": 7, "category": "CUDA Fundamentals",
        "q": "What is a Streaming Multiprocessor (SM) and what resources does it hold?",
        "a": "An SM is the fundamental compute unit of a GPU. Each SM contains: CUDA cores (FP32/INT32 units), Tensor Cores (matrix units), a warp scheduler (issues instructions to warps), a register file (e.g. 64K 32-bit registers on Ada), L1 cache / shared memory (e.g. 128 KB on Ada), and a load/store unit. The RTX 4070 Super has 56 SMs."
    },
    {
        "n": 8, "category": "CUDA Fundamentals",
        "q": "How do you choose block size, and what is the typical recommended value?",
        "a": "Block size must be a multiple of 32 (warp size) and ≤ 1024. 256 is the most common default — it gives good occupancy without excessive register pressure. The optimal value depends on the kernel's register and shared memory usage. You can sweep block sizes (as done in Module 3) and measure bandwidth; for most memory-bound kernels the curve is flat from 128–512."
    },
    {
        "n": 9, "category": "CUDA Fundamentals",
        "q": "What is the kernel launch syntax in CUDA C and in CuPy?",
        "a": "CUDA C: kernel<<<grid, block, shared_mem, stream>>>(args). CuPy RawKernel: kernel((grid,), (block,), args, shared_mem=N). The triple-chevron syntax specifies: number of blocks, threads per block, optional dynamic shared memory bytes, optional stream. CuPy mirrors this with tuples for grid and block dimensions."
    },
    {
        "n": 10, "category": "CUDA Fundamentals",
        "q": "What is the difference between cudaMemcpy and cudaMemcpyAsync?",
        "a": "cudaMemcpy blocks the CPU until the transfer completes. cudaMemcpyAsync enqueues the transfer in a CUDA stream and returns immediately — the CPU can do other work while the DMA engine transfers data. Async transfers require pinned (page-locked) host memory; using pageable memory with Async silently degrades to synchronous behaviour on some drivers."
    },
    {
        "n": 11, "category": "CUDA Fundamentals",
        "q": "What is the compute capability of a GPU and why does it matter?",
        "a": "Compute capability (e.g. 8.9 for Ada) is a version number encoding supported hardware features: 8.x = Ampere/Ada, 7.x = Volta/Turing, 6.x = Pascal. It determines: max threads/block, shared mem size, supported instructions (e.g. __shfl_sync appeared in 3.0, Tensor Cores in 7.0, FP8 in 8.9). You target a minimum compute capability when compiling with nvcc -arch=sm_89."
    },
    {
        "n": 12, "category": "CUDA Fundamentals",
        "q": "What is CUDA occupancy and how is it calculated?",
        "a": "Occupancy = active warps on SM / maximum warps per SM. The RTX 4070 Super supports 48 warps/SM (1536 threads). Active warps are limited by three resources: (1) threads: floor(max_threads_per_SM / threads_per_block) × warps_per_block; (2) shared memory: floor(shmem_per_SM / shmem_per_block); (3) registers: floor(regs_per_SM / regs_per_block). The minimum of these three limits sets actual occupancy."
    },
    {
        "n": 13, "category": "CUDA Fundamentals",
        "q": "What are CUDA streams and how do they enable concurrency?",
        "a": "A CUDA stream is an ordered queue of GPU operations (kernels, memcopies). Operations in different streams can execute concurrently — a kernel in stream 1 can overlap with a H2D transfer in stream 2, as long as hardware resources allow. The default stream (stream 0) is synchronised with all other streams. Non-blocking streams (cudaStreamNonBlocking) don't synchronise with stream 0."
    },
    {
        "n": 14, "category": "CUDA Fundamentals",
        "q": "What is the purpose of cudaDeviceSynchronize() and when should you avoid it?",
        "a": "cudaDeviceSynchronize() blocks the CPU until all previously issued GPU operations complete. It is useful for timing and debugging. Avoid it in hot paths because it stalls the CPU-GPU pipeline — use stream-level synchronisation (cudaStreamSynchronize) instead to only wait for a specific stream, allowing other streams to continue."
    },
    {
        "n": 15, "category": "CUDA Fundamentals",
        "q": "What is cooperative groups in CUDA and what problem does it solve?",
        "a": "Cooperative groups (CUDA 9+) provide explicit synchronisation primitives beyond the block boundary. You can synchronise at warp level (cg::coalesced_threads()), block level (cg::this_thread_block()), or grid level (cg::this_grid() — requires cudaLaunchCooperativeKernel). This solves the limitation that __syncthreads() only synchronises within a block, enabling algorithms like global reductions in a single kernel pass."
    },

    # ── Memory Management (16–30) ─────────────────────────────────────────────
    {
        "n": 16, "category": "Memory Management",
        "q": "List the CUDA memory types from fastest to slowest and their typical sizes.",
        "a": "Registers (~fastest, per-thread, 64K 32-bit regs/SM), Shared memory / L1 (~19 TB/s, 128 KB/SM on Ada, shared within block), L2 cache (~10 TB/s, 36 MB on Ada), Global memory (GDDR6X/HBM, ~672 GB/s on 4070 Super), Pinned host memory (~12 GB/s over PCIe 4), Pageable host memory (~6 GB/s, staging copy penalty)."
    },
    {
        "n": 17, "category": "Memory Management",
        "q": "What is memory coalescing and what access pattern achieves it?",
        "a": "Coalescing merges the 32 memory accesses of a warp into as few transactions as possible. It is achieved when consecutive threads access consecutive memory addresses (stride-1). Each 128-byte cache line is loaded once for all 32 threads. Strided access (stride > 1) breaks coalescing — stride-2 doubles transactions, stride-32 uses 32× more bandwidth. Always ensure array-of-structs layouts are changed to struct-of-arrays when accessed in parallel."
    },
    {
        "n": 18, "category": "Memory Management",
        "q": "What is pinned memory and how does it speed up transfers?",
        "a": "Pinned (page-locked) memory is host memory that the OS guarantees will never be swapped out. This allows the GPU DMA engine to transfer directly between pinned host memory and device memory without a staging copy through a kernel-managed bounce buffer. Result: ~2× higher H2D/D2H bandwidth vs pageable. In CuPy: cp.cuda.alloc_pinned_memory(size). Downside: pinned memory is limited and over-allocation degrades system performance."
    },
    {
        "n": 19, "category": "Memory Management",
        "q": "What is Unified Memory (cudaMallocManaged) and when should you use it?",
        "a": "Unified Memory presents a single pointer accessible from both CPU and GPU. The CUDA driver automatically migrates pages on demand (page faults). Use it for: prototyping (zero boilerplate), sparse irregular access patterns. Avoid for: large arrays with repeated full scans (explicit transfers are faster), latency-sensitive code (page faults cause unpredictable stalls). Always prefetch with memPrefetchAsync before heavy kernel access. In CuPy: cp.cuda.ManagedMemory."
    },
    {
        "n": 20, "category": "Memory Management",
        "q": "What is the Roofline model and how do you use it to diagnose a kernel?",
        "a": "The Roofline model plots attainable performance (GFLOP/s) against arithmetic intensity (FLOP/byte). Two ceilings: peak compute (horizontal) and peak bandwidth × AI (diagonal). A kernel left of the ridge point is memory-bound — optimise memory access. A kernel right of the ridge point is compute-bound — optimise instruction throughput. To use: measure kernel time, calculate FLOPs and bytes transferred, plot point on roofline chart and compare to the ceilings."
    },
    {
        "n": 21, "category": "Memory Management",
        "q": "What is a shared memory bank conflict and how do you avoid it?",
        "a": "Shared memory is divided into 32 banks (one per thread in a warp). If two threads in a warp access the same bank simultaneously (but different addresses), their accesses serialise — a bank conflict. Avoid by padding arrays: e.g. __shared__ float tile[32][33] instead of [32][32]. The extra column ensures each row starts in a different bank. Broadcast (all threads read same address) is conflict-free."
    },
    {
        "n": 22, "category": "Memory Management",
        "q": "What is arithmetic intensity and how do you calculate it for a SAXPY kernel?",
        "a": "Arithmetic intensity = FLOPs / bytes of memory traffic. For SAXPY (y = α·x + y): FLOPs = 2N (one multiply, one add per element). Bytes = 3N × 4 (read x, read y, write y). AI = 2N / (12N) ≈ 0.17 FLOP/byte. This is very low — SAXPY is firmly memory-bound. The Roofline model confirms that adding more compute units would not help; only higher bandwidth matters."
    },
    {
        "n": 23, "category": "Memory Management",
        "q": "How does L1/shared memory carve-up work on Ada GPUs, and can you control it?",
        "a": "Each SM has 128 KB of on-chip SRAM split between L1 cache and shared memory. On Ada you can set the preferred split via cudaFuncSetAttribute(kernel, cudaFuncAttributePreferredSharedMemoryCarveout, percent). Large shared memory (e.g. 96 KB) helps tiled GEMM. Large L1 (e.g. 32+ KB) helps irregular access patterns like stencils. The split must be set before the kernel launch."
    },
    {
        "n": 24, "category": "Memory Management",
        "q": "What is the difference between local memory and shared memory in CUDA?",
        "a": "Shared memory is explicitly managed on-chip SRAM shared by all threads in a block — fast (~19 TB/s). Local memory is per-thread storage that physically resides in global DRAM when registers are insufficient (register spilling) — slow (~672 GB/s). Despite the name 'local', it is not local in the hardware sense. Compilers spill to local memory when a kernel uses more registers than available; check with --ptxas-options=-v."
    },
    {
        "n": 25, "category": "Memory Management",
        "q": "What is memPrefetchAsync and how does it improve Unified Memory performance?",
        "a": "memPrefetchAsync(ptr, size, device_id, stream) tells the CUDA driver to migrate pages to the target device proactively, before a kernel touches them. Without prefetch, the first kernel access triggers a page fault and the driver migrates pages reactively — one page at a time, causing kernel stalls. With prefetch, the migration completes before the kernel starts, eliminating fault-induced latency. Can improve performance by 10× for large UM arrays."
    },
    {
        "n": 26, "category": "Memory Management",
        "q": "What is constant memory in CUDA and when is it beneficial?",
        "a": "Constant memory is a 64 KB read-only region cached in a dedicated constant cache. It is declared with __constant__ and set from the host with cudaMemcpyToSymbol. It is highly efficient when all threads in a warp read the same address (broadcast) — it serves all 32 threads in one transaction. If threads read different addresses, accesses serialise. Ideal for: kernel parameters, lookup tables, filter coefficients that all threads share."
    },
    {
        "n": 27, "category": "Memory Management",
        "q": "How does register pressure affect occupancy and what can you do about it?",
        "a": "Each SM has a fixed register file (64K 32-bit registers on Ada). If a kernel uses many registers per thread, fewer threads can reside on the SM simultaneously, reducing occupancy. To reduce register pressure: use __launch_bounds__(maxThreadsPerBlock, minBlocksPerSM) to hint to the compiler, restructure code to reuse variables, use shared memory for intermediate values, or use -maxrregcount=N with nvcc to cap registers (may cause spilling to local memory)."
    },
    {
        "n": 28, "category": "Memory Management",
        "q": "What is the difference between synchronous and asynchronous memory copies in CUDA streams?",
        "a": "Synchronous (cudaMemcpy): blocks CPU until complete, can be used with any host memory. Asynchronous (cudaMemcpyAsync): enqueues into a stream, CPU returns immediately, requires pinned host memory. With async copies you can overlap: H2D transfer in stream 0, kernel execution in stream 1, D2H transfer in stream 2 — all at the same time. This is the foundation of pipelined GPU programming."
    },
    {
        "n": 29, "category": "Memory Management",
        "q": "What is L2 cache persistence on Ampere/Ada and when is it useful?",
        "a": "Ampere+ (A100, RTX 30/40xx) support L2 cache persistence: you can mark a memory region as 'persisting' so it stays in L2 across kernel launches, and other data as 'streaming' (evicted first). Use cudaStreamAttrValue with accessPolicyWindow. Useful when a hot dataset (e.g. a weight matrix used by many kernels) fits in L2 and you want to avoid re-fetching from DRAM on every kernel. RTX 4070 Super has 36 MB L2."
    },
    {
        "n": 30, "category": "Memory Management",
        "q": "Explain the memory transfer path for pageable vs pinned host memory.",
        "a": "Pageable: CPU allocates via malloc/numpy. Before GPU DMA can transfer, the CUDA driver must: (1) lock pages temporarily, (2) copy to a pinned staging buffer, (3) DMA from staging buffer to device. This double-copy halves achievable bandwidth. Pinned: page-locked at allocation time, so DMA engine reads directly from host memory in a single pass. Pinned also enables cudaMemcpyAsync — the DMA engine can access the pages without OS intervention."
    },

    # ── Warp-Level Programming (31–45) ────────────────────────────────────────
    {
        "n": 31, "category": "Warp-Level Programming",
        "q": "What is warp divergence and what is its performance cost?",
        "a": "Warp divergence occurs when threads in a warp take different code paths (e.g. if/else based on threadIdx). The hardware executes both paths serially, masking inactive threads each time. Cost: if a warp has 2 divergent paths with equal work, execution time doubles. With N mutually exclusive paths, cost is N×. Divergence across warps (based on blockIdx) is free — all threads in that warp agree."
    },
    {
        "n": 32, "category": "Warp-Level Programming",
        "q": "Name three strategies to eliminate or reduce warp divergence.",
        "a": "1. Warp-uniform branches: branch on blockIdx or warp_id = threadIdx.x/32 so all 32 threads agree. 2. Branchless arithmetic: replace if/else with multiplication — selector = (float)(cond); result = selector * a + (1-selector) * b. 3. Sorting/partitioning: reorder work so similar-path elements end up in the same warp (e.g. sort by branch type before launching). 4. Predication: compiler sometimes converts short branches to conditional moves automatically."
    },
    {
        "n": 33, "category": "Warp-Level Programming",
        "q": "What are warp shuffle instructions? Give two examples.",
        "a": "Shuffle intrinsics exchange values between threads in a warp directly via registers — no shared memory, no synchronisation barrier needed. Examples: __shfl_down_sync(mask, val, delta): thread i reads val from thread i+delta. Used for reduction (sum all lanes). __shfl_up_sync(mask, val, delta): thread i reads from thread i-delta. Used for prefix scan. The mask (0xffffffff = all lanes active) specifies participating threads. Faster than shared memory for intra-warp communication."
    },
    {
        "n": 34, "category": "Warp-Level Programming",
        "q": "Walk through a warp-level reduction using __shfl_down_sync.",
        "a": "Each thread starts with its own value v. In 5 steps (delta = 16, 8, 4, 2, 1): v += __shfl_down_sync(0xffffffff, v, delta). After delta=16: lanes 0–15 hold sum of themselves + lanes 16–31. After all steps: lane 0 holds the sum of all 32 values. Total: 5 shuffle instructions vs 5 rounds of shared memory load/store — faster and uses no shared memory."
    },
    {
        "n": 35, "category": "Warp-Level Programming",
        "q": "What is the mask argument in __shfl_*_sync functions?",
        "a": "The mask is a 32-bit integer where bit i=1 means lane i participates in the shuffle. 0xffffffff means all 32 lanes participate (the common case). Using a partial mask is needed when not all threads in a warp are active (e.g. after a divergent branch) or when implementing sub-warp operations. The hardware uses the mask to validate that the source lane is also participating — reading from an inactive lane is undefined behaviour."
    },
    {
        "n": 36, "category": "Warp-Level Programming",
        "q": "What are Tensor Cores and what operation do they accelerate?",
        "a": "Tensor Cores are dedicated matrix-multiply units introduced in Volta (2017). They perform D = A×B + C on small tiles in a single warp instruction. On Ada (RTX 40xx): FP16 tiles of 16×16×16, delivering ~145 TFLOP/s vs ~60 TFLOP/s for FP32 CUDA cores. They support FP16, BF16, TF32, INT8, and FP8. In practice, access them via cuBLAS (cp.matmul on float16 input) or WMMA API. Alignment to multiples of 16 is required for peak throughput."
    },
    {
        "n": 37, "category": "Warp-Level Programming",
        "q": "How do you ensure cp.matmul uses Tensor Cores in CuPy?",
        "a": "Two conditions: (1) input dtype must be float16 or bfloat16, (2) matrix dimensions should be multiples of 16 (8 minimum). CuPy routes matmul through cuBLAS which automatically selects Tensor Core kernels when these conditions are met. You can verify by comparing TFLOP/s against the theoretical peak — FP16 should be ~2–4× faster than FP32 for large matrices. TF32 mode (default in PyTorch Ampere+) also uses Tensor Cores with FP32 inputs."
    },
    {
        "n": 38, "category": "Warp-Level Programming",
        "q": "What are CUDA Graphs and what problem do they solve?",
        "a": "Each kernel launch has CPU-side overhead of ~5–20 µs (Python → driver). For pipelines of many short kernels this overhead dominates. CUDA Graphs let you capture a sequence of operations once (begin_capture → ... → end_capture) and replay the entire graph with a single driver call (~1–5 µs). Speedup is most dramatic for pipelines of short kernels (< 50 µs each). Graphs cannot capture dynamic-shape operations or CPU-GPU synchronisation points."
    },
    {
        "n": 39, "category": "Warp-Level Programming",
        "q": "What limitations do CUDA Graphs have?",
        "a": "Graphs cannot contain: host callbacks (cudaLaunchHostFunc inside capture works but with restrictions), dynamic parallelism launched from CPU, cudaStreamSynchronize inside the captured region, or operations whose parameters change between replays (e.g. different array pointers). Shape must be fixed at capture time. CPU-side Python logic (if/else on results) cannot be captured. You must re-instantiate the graph if parameters change, which is expensive."
    },
    {
        "n": 40, "category": "Warp-Level Programming",
        "q": "What is occupancy and why is 100% occupancy not always optimal?",
        "a": "Occupancy = active warps / max warps per SM. Higher occupancy gives the warp scheduler more warps to switch to when one stalls on memory, hiding latency. But 100% occupancy is not always optimal because: (1) fewer registers per thread may force compiler to spill to slow local memory, (2) small blocks reduce instruction-level parallelism (ILP) within a warp, (3) a compute-bound kernel saturates throughput at 50% occupancy. The roofline model tells you whether you need more occupancy."
    },
    {
        "n": 41, "category": "Warp-Level Programming",
        "q": "What is __launch_bounds__ and how does it help?",
        "a": "__launch_bounds__(maxThreadsPerBlock, minBlocksPerSM) is a per-kernel hint to the compiler. It tells the compiler the maximum block size you will ever use and how many blocks you want per SM. The compiler uses this to constrain register allocation — if it knows max 256 threads/block and wants 4 blocks/SM, it limits registers to 64K/4/256 = 64 per thread. This can improve occupancy at the cost of possibly slightly slower per-thread code."
    },
    {
        "n": 42, "category": "Warp-Level Programming",
        "q": "What is the difference between __shfl_sync and __shfl_xor_sync?",
        "a": "__shfl_sync(mask, val, srcLane): all threads read from a specific lane (srcLane) — a broadcast. __shfl_xor_sync(mask, val, laneMask): thread i reads from lane i XOR laneMask — creates a butterfly exchange pattern. XOR shuffle is the primitive for butterfly reductions and is used in GPU bitonic sort. For reduction, __shfl_down_sync is simpler. For FFT butterfly or sorting networks, __shfl_xor_sync is the natural choice."
    },
    {
        "n": 43, "category": "Warp-Level Programming",
        "q": "What is an inclusive vs exclusive prefix scan?",
        "a": "Inclusive scan: output[i] = sum(input[0..i]). output = [a, a+b, a+b+c, ...]. Exclusive scan: output[i] = sum(input[0..i-1]), output[0] = identity (0 for sum). output = [0, a, a+b, ...]. Inclusive scan can be computed with __shfl_up_sync in O(log2(32)) = 5 steps. Exclusive is the inclusive scan shifted right by one. Prefix scan underlies: stream compaction, radix sort, histogram normalisation."
    },
    {
        "n": 44, "category": "Warp-Level Programming",
        "q": "What is the ridge point in the Roofline model?",
        "a": "Ridge point = Peak FLOP/s ÷ Peak Bandwidth (in FLOP/byte). For an RTX 4070 Super: ~60,000 GFLOP/s ÷ ~672 GB/s ≈ 89 FLOP/byte. Kernels with arithmetic intensity below this are memory-bound; above it are compute-bound. SAXPY (0.17 F/B) and reductions (0.25 F/B) are far left; GEMM (matrix size/2 F/B) is far right. The ridge point is your target: kernels at or above it fully utilise the compute hardware."
    },
    {
        "n": 45, "category": "Warp-Level Programming",
        "q": "What is mixed-precision training and why does it use FP16 for weights but FP32 for gradients?",
        "a": "Mixed precision stores weights and activations in FP16 (half bandwidth, Tensor Core eligible, 2× memory vs FP32) but accumulates gradients and updates in FP32 (avoids underflow/overflow in small gradient values). A master FP32 copy of weights is maintained and used for the optimiser update, then cast back to FP16 for the forward pass. This gives near-FP32 accuracy with ~2× memory savings and ~2–4× compute speedup from Tensor Cores."
    },

    # ── Graphics Compiler & Testing (46–57) ───────────────────────────────────
    {
        "n": 46, "category": "Graphics Compiler & Testing",
        "q": "What does a shader compiler do and what are its key stages?",
        "a": "A shader compiler translates high-level shader code (GLSL/HLSL) into GPU machine code. Key stages: (1) Frontend: parse source, build AST. (2) IR lowering: convert to an intermediate representation (e.g. SPIR-V, DXIL, or NVIDIA's own IR). (3) Optimisation passes: dead code elimination, constant folding, vectorisation, loop unrolling. (4) Register allocation: map virtual registers to physical. (5) Code generation: emit PTX or SASS (native ISA). (6) Scheduling: reorder instructions to hide latency."
    },
    {
        "n": 47, "category": "Graphics Compiler & Testing",
        "q": "What is SPIR-V and why is it important for cross-platform GPU programming?",
        "a": "SPIR-V is a binary intermediate representation (IR) standardised by Khronos Group, used by Vulkan, OpenCL 2.1+, and OpenGL (via extension). It acts as a portable IR: shader compilers (glslang, DXC) emit SPIR-V, and driver-side compilers (NVIDIA's NGX, AMD's ACO) consume it and generate native ISA. This separates front-end compilation (done once, offline) from back-end optimisation (done by the driver). NVIDIA's Vulkan driver accepts SPIR-V and JIT-compiles to SASS."
    },
    {
        "n": 48, "category": "Graphics Compiler & Testing",
        "q": "What is the difference between PTX and SASS?",
        "a": "PTX (Parallel Thread eXecution) is NVIDIA's virtual ISA — an assembly-like IR that is architecture-independent. PTX is compiled to SASS at driver install time or first run (JIT). SASS (Streaming ASSembler) is the actual binary ISA for a specific GPU architecture (e.g. sm_89 for Ada). PTX provides forward compatibility: code compiled to PTX once runs on future GPUs via JIT. SASS provides the lowest-level optimisation but is architecture-specific and undocumented."
    },
    {
        "n": 49, "category": "Graphics Compiler & Testing",
        "q": "How would you design a test plan for a new shader compiler feature?",
        "a": "1. Functional correctness: unit tests for each compiler stage (parser, IR transforms, codegen), end-to-end rendering tests comparing output pixels against a reference renderer. 2. Negative tests: invalid shader inputs, edge cases (empty shader, huge registers, recursion if unsupported). 3. Regression suite: ensure existing conformance suite (OpenGL CTS, Vulkan CTS) still passes. 4. Performance tests: compile-time benchmarks, runtime shader execution benchmarks. 5. Fuzzing: generate random syntactically valid shaders and check for crashes/hangs. 6. Cross-driver comparison: compare output against reference implementation."
    },
    {
        "n": 50, "category": "Graphics Compiler & Testing",
        "q": "What is the difference between JIT and AOT compilation in GPU driver context?",
        "a": "AOT (Ahead-Of-Time): compilation happens before runtime — e.g. offline shader pre-compilation, CUDA fatbinaries bundled in the app. Fast runtime start, no compile latency during gameplay/inference. JIT (Just-In-Time): compilation happens at runtime — e.g. PTX → SASS by the NVIDIA driver on first kernel run. Provides forward compatibility and can optimise for the actual GPU present. CUDA uses a hybrid: PTX is AOT-compiled from source, then JIT-compiled to SASS by the driver cache."
    },
    {
        "n": 51, "category": "Graphics Compiler & Testing",
        "q": "What rendering APIs does NVIDIA's driver support, and what are their differences?",
        "a": "OpenGL: legacy, driver manages lots of state implicitly, high CPU overhead, widely supported. Vulkan: explicit, application manages synchronisation/memory, lower CPU overhead, more control. DirectX 12 (D3D12): Windows-only explicit API similar to Vulkan. DirectX 11 (D3D11): legacy Windows API similar to OpenGL. OpenCL: general-purpose compute, not primarily rendering. CUDA: NVIDIA-specific compute. Vulkan and D3D12 are the focus for modern high-performance rendering."
    },
    {
        "n": 52, "category": "Graphics Compiler & Testing",
        "q": "What is a regression test suite for a GPU driver and how do you keep it healthy?",
        "a": "A regression suite is a collection of tests that must pass on every driver build to ensure no previously working feature was broken. For GPU drivers: pixel-exact rendering tests, performance benchmarks with pass/fail thresholds, API conformance tests (Vulkan CTS, OpenGL CTS, DirectX WHQL), and game/app smoke tests. Keep it healthy by: tagging flaky tests, enforcing CI gating, retiring obsolete tests, and adding a new test for every bug fix (to prevent recurrence)."
    },
    {
        "n": 53, "category": "Graphics Compiler & Testing",
        "q": "What is shader fuzzing and what bugs does it typically find?",
        "a": "Shader fuzzing generates random (but syntactically valid) shader programs and feeds them to the compiler/driver, checking for: crashes (compiler assertion failures, driver timeouts), hangs (GPU hang detection), incorrect output (compare against a reference), and security issues (out-of-bounds access). Tools like Spirv-Fuzz and GLSLSmith generate fuzz inputs. Typical bugs found: miscompilation of edge-case constructs, use-after-free in compiler passes, register allocator corner cases."
    },
    {
        "n": 54, "category": "Graphics Compiler & Testing",
        "q": "How would you isolate a rendering corruption bug in a complex scene?",
        "a": "Binary reduction: disable half the draw calls, check if bug persists. Repeat until single draw call is isolated. Then: simplify the shader (remove uniform branches, constants), replace textures with solid colours, reduce geometry to a quad. Enable API validation layers (Vulkan validation, D3D debug layer) to catch API misuse. Compare output between driver versions (git bisect equivalent). Use RenderDoc or NSight Graphics to capture and replay the frame and inspect each draw call's output."
    },
    {
        "n": 55, "category": "Graphics Compiler & Testing",
        "q": "What is GPU performance profiling and which NVIDIA tools support it?",
        "a": "GPU profiling measures where time is spent in a GPU workload: kernel occupancy, memory bandwidth utilisation, instruction throughput, warp stall reasons. NVIDIA tools: Nsight Compute (per-kernel roofline, source-to-assembly correlation, stall analysis), Nsight Graphics (frame-level GPU timeline, API trace, render target inspection), Nsight Systems (CPU+GPU system-level timeline, stream concurrency). nvprof is deprecated. For CUDA: use cp.cuda.Event or CUDA events API for lightweight timing."
    },
    {
        "n": 56, "category": "Graphics Compiler & Testing",
        "q": "What is the Vulkan Conformance Test Suite (CTS) and what does passing it certify?",
        "a": "The Vulkan CTS (dEQP-VK) is Khronos Group's official conformance test suite with ~750,000 test cases covering every Vulkan feature: rendering correctness, precision, synchronisation, memory model, compute, ray tracing, etc. Passing the CTS certifies that a driver correctly implements the Vulkan specification and allows the vendor to brand their product as 'Vulkan conformant'. NVIDIA's Vulkan drivers must pass CTS before release."
    },
    {
        "n": 57, "category": "Graphics Compiler & Testing",
        "q": "What statistical methods would you use to detect performance regressions in benchmark results?",
        "a": "For noisy GPU benchmarks: use median over mean (more robust to outliers), measure coefficient of variation (CV = σ/μ) to assess noise. Regression detection: t-test or Mann-Whitney U-test to compare before/after distributions, set a threshold (e.g. >3% slowdown = regression). For continuous monitoring: control charts (Shewhart), CUSUM for detecting shifts. Always warm-up before measurement, run enough samples for statistical power, and account for GPU boost clock variability."
    },

    # ── DL Compiler Verification (58–68) ─────────────────────────────────────
    {
        "n": 58, "category": "DL Compiler Verification",
        "q": "What is a deep learning compiler and name three examples.",
        "a": "A DL compiler takes a high-level model graph (from PyTorch/TensorFlow) and compiles it to optimised GPU code. It applies graph-level optimisations (fusion, layout transforms) and generates target-specific kernels. Examples: TVM (open source, targets CPU/GPU/specialised hardware), XLA (Google, used by JAX/TensorFlow), TensorRT (NVIDIA, inference-focused), torch.compile/TorchInductor (PyTorch 2.0), MLIR-based compilers (IREE, Triton)."
    },
    {
        "n": 59, "category": "DL Compiler Verification",
        "q": "What is operator fusion and why is it important for DL performance?",
        "a": "Operator fusion merges multiple operators into a single kernel, avoiding materialising intermediate tensors in global memory. Example: fusing LayerNorm (mean → variance → normalise → scale+shift) from 5 separate kernels into 1. Benefits: eliminates global memory reads/writes for intermediate values, reduces kernel launch overhead, enables register-level caching of intermediate results. Flash Attention is the canonical example: fusing QK^T, softmax, and ×V into one kernel reduces memory from O(n²) to O(n)."
    },
    {
        "n": 60, "category": "DL Compiler Verification",
        "q": "What is ONNX and why is it useful for model portability?",
        "a": "ONNX (Open Neural Network Exchange) is an open format for representing ML models as a computation graph. It defines a standard set of operators and a protobuf schema. Benefits: train in PyTorch, export to ONNX, deploy with TensorRT/ONNX Runtime/CoreML without rewriting. Limitations: not all dynamic shapes or custom ops are representable, operator sets lag behind framework frontends, control flow (loops, conditionals) is limited. Used as an interchange format between training frameworks and inference backends."
    },
    {
        "n": 61, "category": "DL Compiler Verification",
        "q": "What is quantization, and what are the accuracy/performance tradeoffs between INT8 and FP16?",
        "a": "Quantization maps high-precision weights/activations to lower bit widths. FP16: 16-bit float, 2× memory savings, Tensor Core eligible, minimal accuracy loss (typically < 0.1% on standard tasks). INT8: 8-bit integer, 4× memory savings vs FP32, 2× vs FP16, requires calibration (scale/zero-point per tensor), accuracy loss depends on model (usually < 1% with PTQ, < 0.1% with QAT). INT4 and FP8 are newer: 8× savings but higher accuracy risk. Trade-off: smaller dtype = faster inference, more calibration effort, higher accuracy risk."
    },
    {
        "n": 62, "category": "DL Compiler Verification",
        "q": "What is the difference between post-training quantization (PTQ) and quantization-aware training (QAT)?",
        "a": "PTQ: quantize a pre-trained FP32 model without retraining. Only needs a calibration dataset (~100–1000 samples) to compute activation ranges. Fast (minutes), but accuracy loss can be significant for sensitive models. QAT: simulate quantization during training (fake-quantize nodes in the graph), so the model learns to be robust to quantization error. Requires full training (hours/days), produces much better accuracy, especially for INT8 and INT4. PTQ is the default first attempt; use QAT if PTQ accuracy is insufficient."
    },
    {
        "n": 63, "category": "DL Compiler Verification",
        "q": "How would you verify functional correctness of a DL compiler output?",
        "a": "1. Numerical comparison: run both the reference (PyTorch eager) and compiled output on the same input; compare with atol/rtol tolerances (FP16: atol=1e-2, FP32: atol=1e-5). 2. Statistical comparison: on a calibration dataset, check that mean/max absolute error is below threshold. 3. End-to-end task metrics: run compiled model on a benchmark (ImageNet top-1, GLUE score) and compare to FP32 baseline. 4. Edge cases: test with zero input, large values (overflow), NaN/Inf propagation. 5. Hardware sweep: test across GPU architectures (Pascal, Volta, Ampere, Ada)."
    },
    {
        "n": 64, "category": "DL Compiler Verification",
        "q": "What is TensorRT and what key optimisations does it apply?",
        "a": "TensorRT is NVIDIA's inference SDK. Optimisations: (1) Layer fusion: conv+BN+ReLU → single kernel. (2) Precision calibration: FP32 → FP16/INT8. (3) Kernel auto-selection: benchmark multiple CUDA kernels and pick the fastest for each op+shape. (4) Memory optimisation: tensor lifetime analysis to reuse buffers. (5) Dynamic shapes: build engines for shape ranges. (6) Sparsity: structured 2:4 sparse weight support for Ampere+. TensorRT builds a serialised engine (plan file) from ONNX that is optimised for the specific GPU present."
    },
    {
        "n": 65, "category": "DL Compiler Verification",
        "q": "What is kernel auto-tuning and how does TVM implement it?",
        "a": "Auto-tuning benchmarks many candidate kernel implementations (different tile sizes, loop orderings, unroll factors) and selects the fastest for a given op+shape+hardware combination. TVM's AutoTVM and newer Ansor/MetaSchedule define a search space of schedule transformations, sample configurations, compile and benchmark them on the target device, and use a cost model (learned from measurements) to guide the search. Result: a lookup table of tuned kernels, stored as a JSON file, that TVM uses at compile time."
    },
    {
        "n": 66, "category": "DL Compiler Verification",
        "q": "What is torch.compile and how does it differ from TorchScript?",
        "a": "torch.compile (PyTorch 2.0+) uses TorchDynamo to trace Python bytecode and generate optimised kernels via TorchInductor (which generates Triton or C++ code). It is eager-mode compatible — you just wrap the function. TorchScript is an older approach that requires explicitly annotating code for static graph capture (jit.script or jit.trace); it is more limited (no Python arbitrary control flow) but produces a serialisable model. torch.compile is generally preferred for modern workloads."
    },
    {
        "n": 67, "category": "DL Compiler Verification",
        "q": "What is Triton (OpenAI) and how does it relate to CUDA?",
        "a": "Triton is a Python DSL for writing GPU kernels at a higher level than CUDA C. It works in tile-level abstractions: you describe how to load/compute/store tiles of tensors, and the Triton compiler handles tiling, vectorisation, shared memory management, and PTX generation. It targets NVIDIA (and AMD) GPUs. Used by PyTorch 2.0's TorchInductor to generate fused kernels. Easier than raw CUDA for ML kernels but less flexible for non-tiled compute patterns."
    },
    {
        "n": 68, "category": "DL Compiler Verification",
        "q": "How do you detect performance regressions in a DL compiler CI pipeline?",
        "a": "Collect metrics: latency (ms), throughput (samples/s), memory usage (MB) for a fixed benchmark suite (standard models: ResNet-50, BERT, GPT-2) on a fixed GPU. Compare to baseline (previous commit or pinned reference). Use statistical thresholds: flag >3% latency regression as blocking. Control for noise: warm GPU, pin clocks (nvidia-smi -pm 1, --lock-gpu-clocks), use median of N runs. Archive results in a database, plot trends over time. Alert on first occurrence, require sign-off to accept regression."
    },

    # ── Math Libraries & LLM (69–78) ──────────────────────────────────────────
    {
        "n": 69, "category": "Math Libraries & LLM",
        "q": "What is cuBLAS and what is GEMM?",
        "a": "cuBLAS is NVIDIA's GPU-accelerated implementation of BLAS (Basic Linear Algebra Subprograms). GEMM (GEneral Matrix Multiply) computes C = α·A×B + β·C and is the most important routine — it underlies dense layers, attention, and convolution (via im2col). cuBLAS automatically uses Tensor Cores when inputs are FP16/BF16 and dimensions are aligned. cuBLAS handles batched GEMM (many small independent matrix multiplies), strided GEMM, and various layout combinations (row/col major, transpose)."
    },
    {
        "n": 70, "category": "Math Libraries & LLM",
        "q": "What is Flash Attention and why does it use less memory than standard attention?",
        "a": "Standard attention computes S = QK^T (N×N matrix), then P = softmax(S), then O = PV. For sequence length N, S is O(N²) memory. Flash Attention fuses all three steps into a single kernel using tiled computation: it loads small tiles of Q, K, V into shared memory, computes partial results, and updates a running softmax (online softmax trick). It never materialises the full N×N attention matrix in global memory, reducing memory from O(N²) to O(N). This makes it practical for N > 4096."
    },
    {
        "n": 71, "category": "Math Libraries & LLM",
        "q": "What is KV caching in LLM inference and why is it important?",
        "a": "In autoregressive generation, each new token attends to all previous tokens. Without caching, K and V for each previous token would be recomputed on every generation step (O(n²) compute total). KV cache stores the K and V projections for all past tokens; only the new token's K and V need to be computed at each step. This reduces generation compute from O(n²) to O(n) per step. Downside: KV cache grows with sequence length and number of layers; for a 70B model it can be tens of GBs."
    },
    {
        "n": 72, "category": "Math Libraries & LLM",
        "q": "What is the difference between SGEMM and HGEMM?",
        "a": "SGEMM: Single-precision GEMM, FP32 inputs and outputs, uses CUDA cores. HGEMM: Half-precision GEMM, FP16 inputs (FP32 or FP16 accumulator), uses Tensor Cores. HGEMM is ~2–4× faster on Ada Tensor Cores (145 vs 60 TFLOP/s) and uses half the memory bandwidth. For inference: HGEMM is standard. For training: mixed precision uses FP16 inputs with FP32 accumulation to maintain numerical stability. cuBLAS selects HGEMM automatically when input dtype is float16."
    },
    {
        "n": 73, "category": "Math Libraries & LLM",
        "q": "What is cuFFT and where is it used in deep learning?",
        "a": "cuFFT is NVIDIA's GPU FFT library. It computes Fast Fourier Transforms on GPU with automatic algorithm selection (Cooley-Tukey, Bluestein, mixed-radix). In deep learning: (1) Convolution via FFT (conv-FFT fusion is faster than direct conv for large kernels). (2) Spectral methods in physics-informed neural networks. (3) Audio processing (STFT, mel spectrograms). (4) Signal processing layers. For small 3×3 conv kernels, direct cudnn convolution is usually faster; FFT-based conv wins for kernel sizes > ~7×7."
    },
    {
        "n": 74, "category": "Math Libraries & LLM",
        "q": "What is speculative decoding and how does it improve LLM throughput?",
        "a": "Speculative decoding uses a small fast draft model to generate k candidate tokens, then the large target model verifies all k tokens in a single forward pass (parallel, not sequential). If the target model agrees with draft token i but not i+1, it accepts tokens 0..i and rejects the rest, generating a new token at position i+1. Expected speedup: 2–4× for tasks where the draft model is accurate. The target model's compute per accepted token decreases because parallel verification is cheaper than k sequential autoregressive steps."
    },
    {
        "n": 75, "category": "Math Libraries & LLM",
        "q": "What is the transformer attention mechanism and its computational complexity?",
        "a": "Attention: O = softmax(QK^T / √d_k) × V. Q, K, V are linear projections of the input (shape: batch × heads × seq_len × d_k). QK^T is O(n² × d_k) compute, softmax is O(n²), ×V is O(n² × d_v). Total: O(n²·d) per layer — quadratic in sequence length n. For n=4096, d=128, the attention matrix is 4096×4096×32 heads = 2 GB in FP16. This is why Flash Attention (fused, avoids materialising this) and sliding window attention (sparse patterns) are critical for long sequences."
    },
    {
        "n": 76, "category": "Math Libraries & LLM",
        "q": "What is RAG (Retrieval-Augmented Generation) and how is it implemented efficiently on GPU?",
        "a": "RAG retrieves relevant documents from a vector database at query time and prepends them to the LLM context. Efficient GPU implementation: (1) Embedding: encode query with a small embedding model on GPU (e.g. sentence-transformers via cuBLAS). (2) Similarity search: use cuVS (NVIDIA's GPU vector search library, formerly FAISS-GPU) for ANN (approximate nearest neighbour) search — batched inner product on GPU. (3) Reranking (optional): cross-encoder on GPU. (4) Generation: concatenate retrieved docs + query, run LLM. GPU bottleneck is usually the LLM generation step, not retrieval."
    },
    {
        "n": 77, "category": "Math Libraries & LLM",
        "q": "What is rotary positional encoding (RoPE) and why is it preferred over learned positions?",
        "a": "RoPE encodes position by rotating Q and K vectors in 2D subspaces using sin/cos at different frequencies — it modifies the dot product QK^T so that the result depends only on the relative position (i-j), not absolute positions. Advantages: (1) Generalises to longer sequences than seen during training (with modifications like YaRN or LongRoPE). (2) No learned parameters. (3) Compatible with KV caching (position is applied to Q and K before caching). Used in LLaMA, Mistral, and most modern open LLMs."
    },
    {
        "n": 78, "category": "Math Libraries & LLM",
        "q": "What is batch size and how does it affect GPU utilisation during inference?",
        "a": "Batch size = number of requests processed simultaneously. Effect on inference: small batch (1–4) → memory-bound (weights dominate bandwidth, Tensor Cores underutilised). Large batch → compute-bound (Tensor Cores saturated, throughput maximised). For LLM inference, the prefill phase (processing the prompt, large batch equivalent) is compute-bound; the decode phase (one token per step) is memory-bound because batch size is effectively 1 per request. Continuous batching and PagedAttention allow multiple users to share the decode phase, raising effective batch size."
    },

    # ── Inference & Model Optimization (79–100) ───────────────────────────────
    {
        "n": 79, "category": "Inference & Model Optimization",
        "q": "What is PagedAttention and what problem does it solve?",
        "a": "PagedAttention (vLLM, 2023) manages the KV cache like OS virtual memory. Instead of pre-allocating a contiguous block for the max possible sequence length per request (wasteful), it allocates KV cache in fixed-size pages (blocks) and maps them non-contiguously. Benefits: (1) Eliminates KV cache fragmentation and over-allocation. (2) Enables sharing KV cache between requests with the same prefix (prompt caching). (3) Increases effective batch size by ~2–4× for the same VRAM. This is the key innovation behind vLLM's throughput advantage."
    },
    {
        "n": 80, "category": "Inference & Model Optimization",
        "q": "What is continuous batching in LLM inference servers?",
        "a": "Traditional batching: wait until a fixed batch of N requests arrives, process all in parallel, wait for all to finish. Problem: short sequences finish early but must wait for slow sequences (head-of-line blocking). Continuous batching (iteration-level scheduling): at each generation step, check if any request has finished and immediately insert a new request into the batch. Result: the GPU is never waiting for slow sequences to finish — the batch is always full. vLLM, TGI, and TensorRT-LLM all implement continuous batching."
    },
    {
        "n": 81, "category": "Inference & Model Optimization",
        "q": "What is tensor parallelism and when is it used?",
        "a": "Tensor parallelism splits individual weight matrices across multiple GPUs. For a linear layer Y = XW, split W column-wise across N GPUs: each GPU holds W/N columns, computes X×(W/N), then an all-reduce sums partial results. This is Megatron-LM style TP. Use case: model too large to fit on one GPU, or matrix operations too slow on a single GPU. Requires high-bandwidth inter-GPU connections (NVLink) because of the all-reduce. Typically combined with pipeline parallelism for very large models."
    },
    {
        "n": 82, "category": "Inference & Model Optimization",
        "q": "What is pipeline parallelism and what is a micro-batch?",
        "a": "Pipeline parallelism splits layers across GPUs: GPU 0 runs layers 0–11, GPU 1 runs layers 12–23, etc. A micro-batch is a fraction of the batch that flows through the pipeline. Without micro-batching, GPUs are idle while waiting for the next GPU to finish (pipeline bubble). With micro-batching (GPipe / PipeDream), multiple micro-batches are in-flight simultaneously, keeping all GPUs busy. Pipeline bubble fraction ≈ (N_gpus - 1) / N_microbatches. Use when model depth exceeds single-GPU memory."
    },
    {
        "n": 83, "category": "Inference & Model Optimization",
        "q": "What is structured sparsity (2:4) and how does it work on NVIDIA Ampere+?",
        "a": "2:4 structured sparsity requires that in every contiguous group of 4 weights, exactly 2 are zero. This 50% sparsity pattern is compressible: weights and their indices are stored in compressed form using 50% less memory. NVIDIA Sparse Tensor Core units natively compute the sparse matrix multiply without decompressing, delivering up to 2× speedup for FP16 GEMM vs dense. Requires a fine-tuning step (Straight-Through Estimator) to impose the 2:4 pattern while maintaining accuracy."
    },
    {
        "n": 84, "category": "Inference & Model Optimization",
        "q": "What is FP8 precision and when was it introduced by NVIDIA?",
        "a": "FP8 is an 8-bit floating point format introduced in NVIDIA Hopper (H100, 2022). There are two variants: E4M3 (4 exponent bits, 3 mantissa — larger range) and E5M2 (5 exponent, 2 mantissa — more precision). FP8 GEMM delivers ~2× the throughput of FP16 on Tensor Cores. Challenges: FP8's small dynamic range requires per-tensor or per-channel scaling (unlike FP16 which mostly just works). Used in LLM training (TransformerEngine) and inference (TensorRT-LLM) to push the throughput ceiling further."
    },
    {
        "n": 85, "category": "Inference & Model Optimization",
        "q": "What is knowledge distillation and how does it produce smaller, faster models?",
        "a": "Knowledge distillation trains a small student model to mimic a large teacher model's soft output distribution (logits), not just the hard labels. The student minimises KL divergence between its softmax output and the teacher's. The teacher's soft probabilities carry more information than one-hot labels (e.g. a cat image also has 5% probability of being a dog — structurally meaningful). Result: student models are often 2–10× smaller with < 1% accuracy loss. Used to create DistilBERT, TinyLlama, and most edge-deployed models."
    },
    {
        "n": 86, "category": "Inference & Model Optimization",
        "q": "What is NVIDIA Triton Inference Server?",
        "a": "Triton Inference Server is NVIDIA's open-source model serving framework. It supports: multiple backends (TensorRT, ONNX Runtime, PyTorch, TensorFlow, Python), dynamic batching (collects requests and batches them automatically), concurrent model execution (multiple models on same GPU), model ensembles (chain models in a pipeline), gRPC and HTTP endpoints, model repository for easy deployment. It handles production concerns: health checks, metrics (Prometheus), model versioning, GPU memory management."
    },
    {
        "n": 87, "category": "Inference & Model Optimization",
        "q": "What is the difference between latency and throughput in inference, and what governs each?",
        "a": "Latency: time from request received to response sent (ms). Governed by: model size, sequence length, GPU memory bandwidth (decode is memory-bound), batch size (larger batch = higher latency per request). Throughput: requests (or tokens) processed per second. Governed by: batch size (larger = more GPU utilisation), GPU compute, continuous batching efficiency. They trade off: optimising for throughput (large batches) increases per-request latency. SLAs for production typically specify both: e.g. P99 latency < 500 ms and > 1000 tokens/s."
    },
    {
        "n": 88, "category": "Inference & Model Optimization",
        "q": "What is SmoothQuant and what problem does it address?",
        "a": "SmoothQuant addresses the challenge that LLM activations (not weights) have large per-channel outliers that break INT8 quantization. The insight: migrate the quantization difficulty from activations to weights by scaling: X_smooth = X / s, W_smooth = W × s (where s is a per-channel smoothing factor). Activations become easier to quantize (outliers reduced), weights become slightly harder — but weight quantization is easier because weights are static and can be calibrated carefully. Enables W8A8 (INT8 weights and activations) with < 1% accuracy loss on GPT-class models."
    },
    {
        "n": 89, "category": "Inference & Model Optimization",
        "q": "What is vLLM and what are its key innovations?",
        "a": "vLLM is an LLM inference engine focused on high throughput. Key innovations: (1) PagedAttention: OS-inspired virtual memory for KV cache, eliminating fragmentation and enabling prefix sharing. (2) Continuous batching: iteration-level scheduling, no head-of-line blocking. (3) Efficient CUDA kernels for attention and sampling. (4) Prefix caching: share KV cache for common prefixes (system prompts) across requests. Result: 2–24× higher throughput than naive HuggingFace generate() for the same latency budget."
    },
    {
        "n": 90, "category": "Inference & Model Optimization",
        "q": "What is PyTorch 2.0's torch.export and how does it differ from torch.compile?",
        "a": "torch.compile JIT-compiles Python functions at runtime using tracing — dynamic, supports Python control flow, recompiles on shape changes. torch.export statically captures a model into a portable ExportedProgram (a subset of PyTorch that can be executed without Python) — suitable for deployment without a Python runtime, serialisation, and ahead-of-time compilation with TensorRT or ExecuTorch (edge devices). export requires the model to have no data-dependent control flow or uses torch.cond for it."
    },
    {
        "n": 91, "category": "Inference & Model Optimization",
        "q": "What is model sharding and how is it done across multiple GPUs?",
        "a": "Model sharding splits model parameters across GPUs because the model is too large to fit on one device. Strategies: (1) Tensor parallelism: split weight matrices column/row-wise (Megatron-LM). (2) Pipeline parallelism: split layers across GPUs. (3) ZeRO (DeepSpeed): shard optimizer states, gradients, and/or parameters across data-parallel workers — reduces memory per GPU. (4) Sequence parallelism: shard activations along the sequence dimension for long contexts. Modern frameworks combine all four: e.g. Megatron-LM uses TP + PP + ZeRO-1."
    },
    {
        "n": 92, "category": "Inference & Model Optimization",
        "q": "What is GPTQ and AWQ — how do they differ from standard PTQ?",
        "a": "GPTQ and AWQ are weight-only quantization methods targeting LLMs. Standard PTQ: quantize all weights with a simple min/max scale. GPTQ: uses second-order information (Hessian) to find quantization errors and compensate by adjusting remaining weights (lazy batch updates). More accurate than naive PTQ for W4 (4-bit weights). AWQ (Activation-aware Weight Quantization): identifies salient weights (channels with large activations) and applies a per-channel scale before quantization, protecting the most important weights. AWQ is faster to apply than GPTQ and typically comparable accuracy at W4."
    },
    {
        "n": 93, "category": "Inference & Model Optimization",
        "q": "What is Mixture of Experts (MoE) and what are the inference challenges?",
        "a": "MoE replaces dense feed-forward layers with N expert sub-networks, routing each token to the top-k experts via a learned router. Only k/N experts are active per token — total parameters are large but FLOPs per token are small (e.g. Mixtral 8×7B: 46B parameters but FLOPs of a 12B dense model). Inference challenges: (1) All experts must fit in memory even though only k are active — requires multi-GPU sharding. (2) Load imbalance: popular experts get more tokens, causing GPU idle time. (3) Expert routing is sequential; expert compute can be parallelised with expert parallelism."
    },
    {
        "n": 94, "category": "Inference & Model Optimization",
        "q": "What is kernel fusion in inference and give a concrete example from an LLM.",
        "a": "Kernel fusion merges multiple operations into one GPU kernel, avoiding costly round-trips to global memory. Concrete LLM example: the MLP block computes gate(x) × silu(x) × W2, where gate and silu are separate element-wise ops. Fused: a single kernel loads x once, computes gating and silu in registers, multiplies, and writes once. Flash Attention is the most impactful fusion: Q, K, V loading, QK^T matmul, softmax, ×V, and output are all fused into one kernel, eliminating the O(N²) attention matrix materialisation."
    },
    {
        "n": 95, "category": "Inference & Model Optimization",
        "q": "What is quantization calibration and what happens if calibration data is poor?",
        "a": "Calibration determines the quantization scale (and zero-point for asymmetric INT8) by running a representative dataset through the model and collecting activation statistics (min/max or percentile). If calibration data is unrepresentative: (1) Scales are too narrow → clipping of large activations at runtime → accuracy loss. (2) Scales are too wide → poor use of INT8 range → increased quantization error. Best practice: use 100–1000 samples from the actual deployment distribution, including edge cases. For LLMs: use diverse prompts from the target domain."
    },
    {
        "n": 96, "category": "Inference & Model Optimization",
        "q": "What is Medusa decoding?",
        "a": "Medusa adds multiple extra decoding heads to the LLM that predict tokens at future positions (token+1, token+2, …, token+k) simultaneously using the same base model forward pass. The predictions form a tree of candidate continuations, which is verified and pruned using the original model's probabilities. Unlike speculative decoding (which needs a separate draft model), Medusa's draft heads are lightweight MLP additions trained on top of a frozen base model. Result: 2–3× speedup with minimal accuracy loss and no separate model to manage."
    },
    {
        "n": 97, "category": "Inference & Model Optimization",
        "q": "How would you benchmark an LLM inference system end-to-end?",
        "a": "Measure: (1) Time To First Token (TTFT): latency from request submission to first generated token — dominated by prefill. (2) Inter-Token Latency (ITL): average time between tokens during generation — dominated by decode. (3) End-to-end latency: TTFT + ITL × output_tokens. (4) Throughput: tokens/second across all concurrent users. Vary: prompt length, output length, concurrency level (1, 10, 100 users). Tools: locust, k6 for load testing; vLLM benchmark_serving.py; LM-Evaluation-Harness for accuracy. Pin GPU clocks, warm up, use percentiles (P50, P90, P99) not means."
    },
    {
        "n": 98, "category": "Inference & Model Optimization",
        "q": "What is the difference between data parallelism and model parallelism?",
        "a": "Data parallelism (DP): replicate the full model on each GPU, split the batch — each GPU processes different samples, gradients are all-reduced. Requires the model to fit on a single GPU. Scales well (linear throughput with GPU count). Model parallelism (MP): split the model itself across GPUs — necessary when the model is too large for one GPU. Subtypes: tensor parallelism (split layers), pipeline parallelism (split by layer groups). MP requires inter-GPU communication within each sample's forward/backward pass, requiring NVLink for efficiency. Most large-scale training uses DP + TP + PP combined."
    },
    {
        "n": 99, "category": "Inference & Model Optimization",
        "q": "What is prefix caching and how does it reduce inference cost for shared system prompts?",
        "a": "Prefix caching (also called prompt caching) stores the KV cache for a frequently used prefix (e.g. a long system prompt or few-shot examples) and reuses it across requests. When a new request shares the same prefix, the server skips the prefill compute for that prefix entirely — only the unique suffix is processed. Savings: if a 2000-token system prompt is shared by all users, 2000 tokens of prefill compute (quadratic attention) are eliminated per request. vLLM implements this via its radix tree-based KV cache management."
    },
    {
        "n": 100, "category": "Inference & Model Optimization",
        "q": "What is NCCL and why is it critical for multi-GPU deep learning?",
        "a": "NCCL (NVIDIA Collective Communications Library) provides optimised collective operations for multi-GPU and multi-node training: AllReduce (sum gradients across all GPUs), AllGather (gather tensors from all GPUs), ReduceScatter (reduce + scatter for ZeRO), Broadcast, Barrier. NCCL automatically selects the fastest communication path: NVLink (600 GB/s bidirectional between A100 GPUs on NVSwitch), PCIe, or InfiniBand (network). PyTorch DistributedDataParallel, Megatron-LM, and DeepSpeed all rely on NCCL for their communication backend."
    },
]

print(f"✅ {len(QUESTIONS)} questions loaded across {len(set(q['category'] for q in QUESTIONS))} categories")

✅ 100 questions loaded across 7 categories


In [3]:
def run_quiz_text():
    import random

    categories = ["All"] + sorted(set(q["category"] for q in QUESTIONS))
    filtered = [*QUESTIONS]
    idx = 0

    print("Running text-mode quiz (ipywidgets unavailable).")
    print("After each question: [enter]=next, p=prev, r=random, c=category, q=quit")

    while filtered:
        q = filtered[idx]
        print(f"\nQuestion {idx+1}/{len(filtered)}  [{q['category']}]  Q{q['n']}")
        print(q["q"])
        input("Your answer (optional, press enter to continue): ")
        print(f"Model answer:\n{q['a']}")

        cmd = input("Command: ").strip().lower()
        if cmd == "q":
            print("Quiz ended.")
            return
        if cmd == "p":
            idx = max(0, idx - 1)
            continue
        if cmd == "r":
            idx = random.randrange(len(filtered))
            continue
        if cmd == "c":
            print("Categories:")
            for i, cat in enumerate(categories):
                print(f"  {i}. {cat}")
            sel = input("Pick category number: ").strip()
            if sel.isdigit() and int(sel) < len(categories):
                chosen = categories[int(sel)]
                filtered = QUESTIONS if chosen == "All" else [q for q in QUESTIONS if q["category"] == chosen]
                idx = 0
            else:
                print("Invalid selection; keeping current category.")
            continue

        idx = min(idx + 1, len(filtered) - 1)


def run_quiz():
    if not WIDGETS_AVAILABLE:
        run_quiz_text()
        return
    categories = ["All"] + sorted(set(q["category"] for q in QUESTIONS))
    filtered   = [*QUESTIONS]
    state      = {"idx": 0, "show": False}

    # ── Styles ────────────────────────────────────────────────────────────────
    CARD  = "background:#1e1e2e;border-radius:12px;padding:24px;font-family:monospace;"
    QSTYL = "color:#cdd6f4;font-size:1.1em;line-height:1.6;margin-bottom:12px;"
    ASTYL = "color:#a6e3a1;font-size:1em;line-height:1.6;white-space:pre-wrap;"
    METAC = "color:#6c7086;font-size:0.85em;margin-bottom:12px;"

    # ── Widgets ───────────────────────────────────────────────────────────────
    cat_dd   = widgets.Dropdown(options=categories, description="Category:", layout=widgets.Layout(width="340px"))
    prog_lbl = widgets.HTML()
    q_html   = widgets.HTML()
    ans_area = widgets.Textarea(placeholder="Type your answer here, then click Evaluate or Show Answer…",
                                layout=widgets.Layout(width="100%", height="120px"))
    btn_prev = widgets.Button(description="◀ Prev",   button_style="", layout=widgets.Layout(width="100px"))
    btn_next = widgets.Button(description="Next ▶",   button_style="", layout=widgets.Layout(width="100px"))
    btn_show = widgets.Button(description="Show Answer", button_style="warning", layout=widgets.Layout(width="130px"))
    btn_ai   = widgets.Button(description="⚡ Evaluate with AI", button_style="success",
                               layout=widgets.Layout(width="180px"), disabled=not AI_AVAILABLE)
    btn_rand = widgets.Button(description="🎲 Random",  button_style="info",  layout=widgets.Layout(width="90px"))
    ans_out  = widgets.Output()
    ai_out   = widgets.Output()

    def current_q():
        return filtered[state["idx"]]

    def refresh_question():
        state["show"] = False
        q = current_q()
        prog_lbl.value = (f'<span style="color:#89b4fa;font-size:0.9em">'
                          f'Question {state["idx"]+1} / {len(filtered)} &nbsp;'
                          f'<span style="color:#cba6f7">[{q["category"]}]</span></span>')
        q_html.value   = (f'<div style="{CARD}">'
                          f'<div style="{METAC}">Q{q["n"]}</div>'
                          f'<div style="{QSTYL}"><b>{q["q"]}</b></div>'
                          f'</div>')
        ans_area.value = ""
        ans_out.clear_output()
        ai_out.clear_output()

    def show_answer(_=None):
        state["show"] = True
        q = current_q()
        ans_out.clear_output()
        with ans_out:
            display(HTML(f'<div style="{CARD} margin-top:8px;">'
                         f'<div style="color:#89dceb;font-size:0.8em;margin-bottom:6px;">MODEL ANSWER</div>'
                         f'<div style="{ASTYL}">{q["a"]}</div>'
                         f'</div>'))

    def evaluate_ai(_=None):
        user_ans = ans_area.value.strip()
        q = current_q()
        if not user_ans:
            ai_out.clear_output()
            with ai_out:
                display(HTML('<div style="color:#f38ba8;padding:8px">Please type your answer first.</div>'))
            return
        btn_ai.disabled = True
        btn_ai.description = "Evaluating…"
        ai_out.clear_output()
        with ai_out:
            display(HTML('<div style="color:#89b4fa;padding:8px">⏳ Claude is reviewing your answer…</div>'))
        try:
            prompt = (
                f"You are a senior NVIDIA engineer conducting a technical interview.\n\n"
                f"Question: {q['q']}\n\n"
                f"Model answer: {q['a']}\n\n"
                f"Candidate's answer: {user_ans}\n\n"
                f"Evaluate the candidate's answer. Be concise (3–5 sentences). Cover:\n"
                f"1. What they got right\n"
                f"2. What is missing or incorrect\n"
                f"3. A score: Excellent / Good / Partial / Needs Work\n"
                f"Use plain text, no markdown headers."
            )
            response = _client.messages.create(
                model="claude-sonnet-4-6",
                max_tokens=400,
                messages=[{"role": "user", "content": prompt}]
            )
            feedback = response.content[0].text
            colour   = {"Excellent": "#a6e3a1", "Good": "#a6e3a1",
                        "Partial": "#f9e2af", "Needs Work": "#f38ba8"}
            border   = next((colour[k] for k in colour if k in feedback), "#89b4fa")
            ai_out.clear_output()
            with ai_out:
                display(HTML(
                    f'<div style="{CARD} margin-top:8px; border-left: 4px solid {border};">'
                    f'<div style="color:#cba6f7;font-size:0.8em;margin-bottom:6px;">AI FEEDBACK</div>'
                    f'<div style="color:#cdd6f4;font-size:0.95em;line-height:1.6;white-space:pre-wrap">{feedback}</div>'
                    f'</div>'
                ))
            # Also reveal the model answer after AI evaluation
            show_answer()
        except Exception as e:
            ai_out.clear_output()
            with ai_out:
                display(HTML(f'<div style="color:#f38ba8;padding:8px">Error: {e}</div>'))
        finally:
            btn_ai.disabled = False
            btn_ai.description = "⚡ Evaluate with AI"

    def on_prev(_):
        if state["idx"] > 0:
            state["idx"] -= 1
            refresh_question()

    def on_next(_):
        if state["idx"] < len(filtered) - 1:
            state["idx"] += 1
            refresh_question()

    def on_rand(_):
        import random
        state["idx"] = random.randrange(len(filtered))
        refresh_question()

    def on_cat(change):
        nonlocal filtered
        cat = change["new"]
        filtered = QUESTIONS if cat == "All" else [q for q in QUESTIONS if q["category"] == cat]
        state["idx"] = 0
        refresh_question()

    btn_prev.on_click(on_prev)
    btn_next.on_click(on_next)
    btn_show.on_click(show_answer)
    btn_ai.on_click(evaluate_ai)
    btn_rand.on_click(on_rand)
    cat_dd.observe(on_cat, names="value")

    refresh_question()

    display(widgets.VBox([
        cat_dd,
        prog_lbl,
        q_html,
        ans_area,
        widgets.HBox([btn_prev, btn_next, btn_rand,
                      widgets.HTML("&nbsp;&nbsp;"),
                      btn_show, btn_ai]),
        ans_out,
        ai_out,
    ]))

run_quiz()